# 1. Introdução

**CatBoost aplicado à previsão de churn**

Atividade Final de Aprendizado de Máquina I · FATEC Jundiaí · Ciência de Dados

| Aluna | Professor | Tema |
|:--|:--|:--|
| Mariana Costa de Mello | Mateus Guilherme Fuini | CatBoost |

## O que é o CatBoost

Biblioteca de **gradient boosting** com árvores de decisão, criada pela Yandex e aberta em 2017. O nome vem de *Categorical + Boosting*: a proposta é usar variáveis categóricas direto no modelo, sem codificar antes. É da mesma família do XGBoost e do LightGBM.

## Problemas que ele resolve

| Problema | Como o CatBoost resolve |
|:--|:--|
| Categóricas precisam virar número, e one-hot, ordinal ou target encoding simples têm defeitos | Ordered Target Statistics, dentro do algoritmo |
| *Prediction shift*: o resíduo de um exemplo vem de um modelo que já o viu | Ordered Boosting |
| Overfitting e muito ajuste de hiperparâmetros | Árvores simétricas, regularização L2 e bons valores padrão |
| Predição lenta em árvores grandes | Árvores simétricas funcionam como tabelas indexadas |

## O que é novo em relação à disciplina

- **Boosting**: árvores treinadas em sequência, diferente do *bagging* do Random Forest
- **Categóricas nativas**: sem one-hot no pipeline
- **Árvores simétricas**: estrutura diferente da árvore CART

## Problema escolhido

| Item | Definição |
|:--|:--|
| Base | IBM Telco Customer Churn (7.043 clientes, 19 variáveis) |
| Alvo | `Churn` = 1 se o cliente cancelou |
| Momento da previsão | Foto mensal da base, com o que já se sabe do cliente |
| Uso | Decidir quem recebe ação de retenção |
| Tarefa | Classificação binária desbalanceada |

## Preparação

A mesma preparação é usada em todos os notebooks:

- `TotalCharges` vem como texto e os 11 valores em branco são de clientes com 0 meses de casa, então viram 0
- `customerID` sai porque é só um identificador
- Divisão estratificada em 60% treino, 20% validação e 20% teste, com semente 42

A primeira célula instala as bibliotecas; o `%%capture` esconde as mensagens da instalação.

In [8]:
%%capture
%pip install catboost shap category_encoders

In [9]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.precision", 3)
SEED = 42

df = pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].eq("Yes").astype(int)
cats = X.select_dtypes(exclude="number").columns.tolist()

X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.25, stratify=y_trval, random_state=SEED)

## Visão geral dos dados

Tipos, quantidade de categorias e proporção do alvo.

In [10]:
display(X.describe(include="object").T[["unique", "top", "freq"]])
y.value_counts(normalize=True).rename({0: "Ficou", 1: "Cancelou"})

,unique,top,freq
gender,2,Male,3555
SeniorCitizen,2,No,5901
Partner,2,No,3641
Dependents,2,No,4933
PhoneService,2,Yes,6361
MultipleLines,3,No,3390
InternetService,3,Fiber optic,3096
OnlineSecurity,3,No,3498
OnlineBackup,3,No,3088
DeviceProtection,3,No,3095


Churn
Ficou       0.735
Cancelou    0.265
Name: proportion, dtype: float64

> 16 das 19 variáveis são categóricas, todas com 2 a 4 categorias, e 26,5% dos clientes cancelaram.

## O CatBoost aceita texto

A Regressão Logística do scikit-learn não treina com colunas de texto. O CatBoost só precisa saber quais colunas são categóricas.

In [11]:
from sklearn.linear_model import LogisticRegression

try:
    LogisticRegression().fit(X_train, y_train)
except ValueError as erro:
    print("Regressão Logística:", str(erro).splitlines()[0])

modelo = CatBoostClassifier(iterations=300, random_seed=SEED, verbose=0, cat_features=cats).fit(X_train, y_train)
print(f"CatBoost: ROC-AUC na validação = {roc_auc_score(y_val, modelo.predict_proba(X_val)[:, 1]):.3f}")

Regressão Logística: could not convert string to float: 'Male'
CatBoost: ROC-AUC na validação = 0.837


## Organização do trabalho

| Notebook | Conteúdo |
|:--|:--|
| 1. Introdução | Técnica, problema e dados |
| 2. Fundamentação teórica | Boosting, Ordered TS, Ordered Boosting, árvores simétricas, hiperparâmetros |
| 3. Aplicações práticas | Casos reais e exemplos de regressão e ranqueamento |
| 4. Implementação | Qualidade dos dados, EDA, baselines e primeiro modelo |
| 5. Experimentação | Cinco experimentos |
| 6. Avaliação | Resultado no teste, limiar, calibração e SHAP |
| 7. Análise crítica | Comparação com os modelos da disciplina |
| 8. Conclusão | Resultados, aprendizados e referências |

O conjunto de teste só é usado nos notebooks 6 e 7.